In [1]:
import pandas as pd
import numpy as np
from nocasedict import NocaseDict
import simplejson
import math
from IPython.display import display, HTML

pd.set_option('display.max_columns', None)

In [2]:
def displaywithHeader(df, title) :
    display(HTML(f'<h3>{title}</h3>'))
    display(df.head(3))

# Get Data

### Get Data From IS Spatial Hub

In [3]:
data_real = pd.read_csv("https://data.spatialhub.scot/dataset/9a3728b4-49ea-40af-ab10-fc0305bace84/resource/7ba35197-7ca7-4477-a38b-01fd4180466b/download/lgbf_data_table_real.csv")
data_cash = pd.read_csv("https://data.spatialhub.scot/dataset/9a3728b4-49ea-40af-ab10-fc0305bace84/resource/77c2bc92-ad24-401c-8b53-2160cd12e287/download/lgbf_data_table_cash.csv")

displaywithHeader(data_real,'data_real')
displaywithHeader(data_cash,'data_cash')


,Indicators_Information_Code,LA_Information_LocalAuthority,LA_Data_LGBF_Year,LA_Data_LA_IndicatorReal,LA_Data_LA_Numerator_real,LA_Data_LA_Den_Real,Scotland_Data_Scotland_Indicator_Real,Scotland_Data_Scotland_Num_Real,Scotland_Data_Scotland_Den_Real,FG_Data_FG_Avg_Indicator_Real,FG_Data_FG_Avg_Num_Real,FG_Data_FG_Avg_Den_Real,FG_Data_FamilyGroup
0,C&L01,Aberdeen City,2010-11,0.4777,922.0499,1922292.0,5.1386,233685.3624,45459818.0,5.033650,12513.353075,2200418.875,Urban
1,C&L01,Aberdeen City,2011-12,1.0771,2202.3778,2045051.0,4.5777,220407.8509,48202343.0,4.871738,11905.276475,2355970.750,Urban
2,C&L01,Aberdeen City,2012-13,5.0708,10981.6476,2163756.0,4.3743,225690.0620,51624697.0,5.095187,13195.947813,2601296.500,Urban


,Indicators_Information_ServiceArea,Indicators_Information_Code,Indicators_Information_Title,LA_Information_LocalAuthority,LA_Data_LGBF_Year,LA_Data_LA_IndicatorCash,LA_Data_LA_Numerator_Cash,LA_Data_LA_Den_Cash,Scotland_Data_Scotland_Indicator_Cash,Scotland_Data_Scotland_Num_Cash,Scotland_Data_Scotland_Den_Cash,FG_Data_FG_Avg_Indicator_Cash,FG_Data_FG_Avg_Num_Cash,FG_Data_FG_Avg_Den_Cash,LA_Data_FamilyGroup,Indicators_Information_Category,Indicators_Information_Unit
0,Adult Social Care Services,SW01,Home care costs per hour for people aged 65 or...,Aberdeen City,2010-11,20.12,13516.0,671922.16,20.14,435041.000,21602215.96,21.14625,10960.0000,548521.155,Least Deprived,Financial,Pounds
1,Adult Social Care Services,SW01,Home care costs per hour for people aged 65 or...,Aberdeen City,2011-12,19.78,13902.0,702913.12,19.77,435328.581,22016337.20,22.61250,11730.5000,550521.920,Least Deprived,Financial,Pounds
2,Adult Social Care Services,SW01,Home care costs per hour for people aged 65 or...,Aberdeen City,2012-13,26.83,16345.0,609256.96,20.47,456571.260,22308242.84,24.76250,13007.4075,601998.865,Least Deprived,Financial,Pounds


### Load Family Group and Indicator Info Datasets

In [4]:
info = pd.read_csv('Data Files\\Indicator Information.csv')
fg = pd.read_csv('Data Files\\Family Groups.csv')

displaywithHeader(info,'info')
displaywithHeader(fg,'familygroups')

,Title,Code,Code_Sortable,ReportingPeriod,MeasureType,NumberFormat,YMin,YMax,ISCategory,Committee,FamilyGrouping,StirlingService,Ranking_Type,NumberFormat_NoText,Source,Numerator_Correct,Denominator_Correct,Numerator_Match,Denominator_Match,Numerator_Multipier,Denominator_Multiplier,Ranking_GoldilocksMidpoint,NumberFormat_Axis,Format_Python,FormatAxis_Python,AdditionalAxisDenominator_Python,FormatAxis_Plotly_Prefix,FormatAxis_Plotly_Suffix,ImgPxlWidth_Plotly,ImgPxlHeight_Plotly,SubGroup_PythonReport,YMin_Plotly,YMax_Plotly,OData__ColorTag,Group_PythonReport
0,Net Cost of Waste Collection per Premises,ENV01a,ENV 01a,Annual,Cost,'£ '0,40.0,NaN,Environmental Services,Environment and Housing,"Environmental, Culture & Leisure, Economic Dev...",Environment & Place ; Waste Services,Ascending,0,For more details on Net Waste Collection Costs...,Waste collection - Net expenditure,Number of Premises for Refuse Collection,Waste collection - Net expenditure (£000s),Number of Premises for Refuse Collection,1000.0,1.0,NaN,'£ '0,£ {:0.0f},",d",1,£,NaN,800,493,Waste & Recycling,NaN,NaN,NaN,Environment
1,Net Cost of Waste Disposal per Premises,ENV02a,ENV 02a,Annual,Cost,'£ '0,60.0,NaN,Environmental Services,Environment and Housing,"Environmental, Culture & Leisure, Economic Dev...",Environment & Place ; Waste Services,Ascending,0,For more details on Net Waste Collection Costs...,Waste disposal - Net expenditure,Number of Premises for Refuse Collection,Waste disposal - Net expenditure (£000s),Number of Premises for Refuse Collection,1000.0,1.0,NaN,'£ '0,£ {:0.0f},",d",1,£,NaN,800,493,Waste & Recycling,70.0,130.0,NaN,Environment
2,The % of Total Household Waste Arising that is...,ENV06,ENV 06,Annual,Percentage,0.00%,40.0,100.0,Environmental Services,Environment and Housing,"Environmental, Culture & Leisure, Economic Dev...",Environment & Place ; Waste Services,Descending,0,Data is available for each council on the SEPA...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0%,{:.1%},",.0%",1,NaN,NaN,800,493,Waste & Recycling,0.4,0.6,NaN,Environment


,Family_Group,Local_Authority,Type
0,Family Group 1,Eilean Siar,"Environmental, Culture & Leisure, Economic Dev..."
1,Family Group 1,Argyll & Bute,"Environmental, Culture & Leisure, Economic Dev..."
2,Family Group 1,Shetland Islands,"Environmental, Culture & Leisure, Economic Dev..."


## Initial Data Transformations

In [5]:
data_cash.columns

Index(['Indicators_Information_ServiceArea', 'Indicators_Information_Code',
       'Indicators_Information_Title', 'LA_Information_LocalAuthority',
       'LA_Data_LGBF_Year', 'LA_Data_LA_IndicatorCash',
       'LA_Data_LA_Numerator_Cash', 'LA_Data_LA_Den_Cash',
       'Scotland_Data_Scotland_Indicator_Cash',
       'Scotland_Data_Scotland_Num_Cash', 'Scotland_Data_Scotland_Den_Cash',
       'FG_Data_FG_Avg_Indicator_Cash', 'FG_Data_FG_Avg_Num_Cash',
       'FG_Data_FG_Avg_Den_Cash', 'LA_Data_FamilyGroup',
       'Indicators_Information_Category', 'Indicators_Information_Unit'],
      dtype='object')

### Split To Rows

In [6]:
# Select Only Necessary Columns
data_real = data_real[
        [
            'LA_Information_LocalAuthority',
            'LA_Data_LGBF_Year',
            'Indicators_Information_Code', 
            'LA_Data_LA_IndicatorReal',
            'LA_Data_LA_Numerator_real',
            'LA_Data_LA_Den_Real',
            'Scotland_Data_Scotland_Indicator_Real',
            'Scotland_Data_Scotland_Num_Real',
            'Scotland_Data_Scotland_Den_Real'
        ]
    ]
data_cash = data_cash[
        [
            'LA_Information_LocalAuthority',
            'LA_Data_LGBF_Year',
            'Indicators_Information_Code', 
            'LA_Data_LA_IndicatorCash',
            'LA_Data_LA_Numerator_Cash',
            'LA_Data_LA_Den_Cash',
            'Scotland_Data_Scotland_Indicator_Cash',
            'Scotland_Data_Scotland_Num_Cash',
            'Scotland_Data_Scotland_Den_Cash'
        ]
    ]

# Rename Columns to Append
data_real = data_real.rename(
                columns = {
                            'Indicators_Information_Code': 'Code',
                            'LA_Information_LocalAuthority': 'LocalAuthority',
                            'LA_Data_LGBF_Year': 'Period',
                            'LA_Data_LA_IndicatorReal' : 'Value',
                            'LA_Data_LA_Numerator_real' : 'Numerator',
                            'LA_Data_LA_Den_Real' : 'Denominator',
                            'Scotland_Data_Scotland_Indicator_Real': "ValueScot",
                            'Scotland_Data_Scotland_Num_Real': "NumeratorScot",
                            'Scotland_Data_Scotland_Den_Real': "DenominatorScot"
                        }
            )
data_cash = data_cash.rename(
                columns = {
                            'Indicators_Information_Code': 'Code',
                            'LA_Information_LocalAuthority': 'LocalAuthority',
                            'LA_Data_LGBF_Year': 'Period',
                            'LA_Data_LA_IndicatorCash' : 'Value',
                            'LA_Data_LA_Numerator_Cash' : 'Numerator',
                            'LA_Data_LA_Den_Cash' : 'Denominator',
                            'Scotland_Data_Scotland_Indicator_Cash': "ValueScot",
                            'Scotland_Data_Scotland_Num_Cash' : "NumeratorScot",
                            'Scotland_Data_Scotland_Den_Cash' : "DenominatorScot"
                        }
            )

# Add Data Type Column
data_real.insert(1,'DataType','Real')
data_cash.insert(1,'DataType','Cash')

# Append Together
data = pd.concat([data_real, data_cash], ignore_index=True)

displaywithHeader(data,'data')

,LocalAuthority,DataType,Period,Code,Value,Numerator,Denominator,ValueScot,NumeratorScot,DenominatorScot
0,Aberdeen City,Real,2010-11,C&L01,0.4777,922.0499,1922292.0,5.1386,233685.3624,45459818.0
1,Aberdeen City,Real,2011-12,C&L01,1.0771,2202.3778,2045051.0,4.5777,220407.8509,48202343.0
2,Aberdeen City,Real,2012-13,C&L01,5.0708,10981.6476,2163756.0,4.3743,225690.0620,51624697.0


### Add in Missing Annual Indicator Data for SW08,ECON 12a and ECON 12b

In [7]:
missing_data = pd.read_csv('Raw Data Files\\MissingData_Real.csv')
data = pd.concat([data,missing_data])

displaywithHeader(data,'data')

C:\Users\lowsona\AppData\Local\Temp\ipykernel_26120\3029833504.py:2: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  data = pd.concat([data,missing_data])


,LocalAuthority,DataType,Period,Code,Value,Numerator,Denominator,ValueScot,NumeratorScot,DenominatorScot
0,Aberdeen City,Real,2010-11,C&L01,0.4777,922.0499,1922292.0,5.1386,233685.3624,45459818.0
1,Aberdeen City,Real,2011-12,C&L01,1.0771,2202.3778,2045051.0,4.5777,220407.8509,48202343.0
2,Aberdeen City,Real,2012-13,C&L01,5.0708,10981.6476,2163756.0,4.3743,225690.0620,51624697.0


### Determine Update Period Types (Annual, Quarterly, Monthly), Create a more Standard Delimiter for Monthly and Quarterly

In [8]:
# Replace Space with ';' as delimiter
data['Period'] = data['Period'].str.replace(' ',';')


# Define Function to Identify DataType
Months = ['January', 'February', 'March', 'April', 'May', 'June', 'July', 'August', 'September', 'October', 'November', 'December']

def funcPeriodType(df) :
    if any(substr in df.Period for substr in Months) :
        return f'{df.DataType}_Monthly'
    elif 'Q' in df.Period :
        return f'{df.DataType}_Quarterly'
    else: 
        return f'{df.DataType}_Annual'

# Evaluate DataType over the Dataframe
data['DataType'] = data.apply(funcPeriodType, axis=1)
displaywithHeader(data,'data')

,LocalAuthority,DataType,Period,Code,Value,Numerator,Denominator,ValueScot,NumeratorScot,DenominatorScot
0,Aberdeen City,Real_Annual,2010-11,C&L01,0.4777,922.0499,1922292.0,5.1386,233685.3624,45459818.0
1,Aberdeen City,Real_Annual,2011-12,C&L01,1.0771,2202.3778,2045051.0,4.5777,220407.8509,48202343.0
2,Aberdeen City,Real_Annual,2012-13,C&L01,5.0708,10981.6476,2163756.0,4.3743,225690.0620,51624697.0


### Fix Value Issues

In [9]:
#Remove NA Rows
data = data[data['Value'].notna()]
displaywithHeader(data,'data')

,LocalAuthority,DataType,Period,Code,Value,Numerator,Denominator,ValueScot,NumeratorScot,DenominatorScot
0,Aberdeen City,Real_Annual,2010-11,C&L01,0.4777,922.0499,1922292.0,5.1386,233685.3624,45459818.0
1,Aberdeen City,Real_Annual,2011-12,C&L01,1.0771,2202.3778,2045051.0,4.5777,220407.8509,48202343.0
2,Aberdeen City,Real_Annual,2012-13,C&L01,5.0708,10981.6476,2163756.0,4.3743,225690.0620,51624697.0


## Separate Scotland Values

In [10]:
data_scot = data[
    [
        'Code',
        'Period',
        'ValueScot',
        'NumeratorScot',
        'DenominatorScot',
        'DataType'
    ]
]

data_scot = data_scot.drop_duplicates()

data_scot = data_scot.rename(
                columns = {
                            'ValueScot' : 'Value',
                            'NumeratorScot' : 'Numerator',
                            'DenominatorScot' : 'Denominator'
                        }
            )

displaywithHeader(data_scot,'data_scot')

,Code,Period,Value,Numerator,Denominator,DataType
0,C&L01,2010-11,5.1386,233685.3624,45459818.0,Real_Annual
1,C&L01,2011-12,4.5777,220407.8509,48202343.0,Real_Annual
2,C&L01,2012-13,4.3743,225690.0620,51624697.0,Real_Annual


## Transform Dataframes using Info and Family Group Data

### Merge Info and Family Group Dataframes

In [11]:
# Merge Dataframes
data = data.merge(info, how='left', on='Code')
data = data.merge(fg, how='left', right_on=['Local_Authority', 'Type'], left_on=['LocalAuthority', 'FamilyGrouping'])

data_scot = data_scot.merge(info, how='left', on='Code')

# Select Only Relevant Columns
data = data[[
    'Code_Sortable',
    'LocalAuthority',
    'Period',
    'Value',
    'Numerator',
    'Denominator',
    'FamilyGrouping',
    'Ranking_Type',
    'Ranking_GoldilocksMidpoint',
    'Family_Group',
    'Numerator_Multipier',
    'Denominator_Multiplier',
    'DataType'
]]

data_scot = data_scot[[
    'Code_Sortable',
    'Period',
    'Value',
    'Numerator',
    'Denominator',
    'Ranking_Type',
    'Ranking_GoldilocksMidpoint',
    'Numerator_Multipier',
    'Denominator_Multiplier',
    'DataType'
]]

# Rename Columns Appropriately
data = data.rename(columns={
    'Code_Sortable': 'Code',
    'FamilyGrouping': 'FG_Type'
})

data_scot = data_scot.rename(columns={
    'Code_Sortable': 'Code',
})

displaywithHeader(data_scot,'data_scot')
displaywithHeader(data,'data')

,Code,Period,Value,Numerator,Denominator,Ranking_Type,Ranking_GoldilocksMidpoint,Numerator_Multipier,Denominator_Multiplier,DataType
0,C&L 01,2010-11,5.1386,233685.3624,45459818.0,Ascending,NaN,1000.0,1.0,Real_Annual
1,C&L 01,2011-12,4.5777,220407.8509,48202343.0,Ascending,NaN,1000.0,1.0,Real_Annual
2,C&L 01,2012-13,4.3743,225690.0620,51624697.0,Ascending,NaN,1000.0,1.0,Real_Annual


,Code,LocalAuthority,Period,Value,Numerator,Denominator,FG_Type,Ranking_Type,Ranking_GoldilocksMidpoint,Family_Group,Numerator_Multipier,Denominator_Multiplier,DataType
0,C&L 01,Aberdeen City,2010-11,0.4777,922.0499,1922292.0,"Environmental, Culture & Leisure, Economic Dev...",Ascending,NaN,Family Group 4,1000.0,1.0,Real_Annual
1,C&L 01,Aberdeen City,2011-12,1.0771,2202.3778,2045051.0,"Environmental, Culture & Leisure, Economic Dev...",Ascending,NaN,Family Group 4,1000.0,1.0,Real_Annual
2,C&L 01,Aberdeen City,2012-13,5.0708,10981.6476,2163756.0,"Environmental, Culture & Leisure, Economic Dev...",Ascending,NaN,Family Group 4,1000.0,1.0,Real_Annual


### Apply Numerator and Denominator Multipliers as Required

In [12]:
# Multiply using Info Dataframe Columns
data['Numerator'] = data['Numerator'] * data['Numerator_Multipier']
data['Denominator'] = data['Denominator'] * data['Denominator_Multiplier']

data_scot['Numerator'] = data_scot['Numerator'] * data_scot['Numerator_Multipier']
data_scot['Denominator'] = data_scot['Denominator'] * data_scot['Denominator_Multiplier']

# Remove Multiplier Columns
data = data.drop(columns= ['Numerator_Multipier','Denominator_Multiplier'])
scotValues = data_scot.drop(columns= ['Numerator_Multipier','Denominator_Multiplier'])

displaywithHeader(data_scot,'data_scot')
displaywithHeader(data,'data')

,Code,Period,Value,Numerator,Denominator,Ranking_Type,Ranking_GoldilocksMidpoint,Numerator_Multipier,Denominator_Multiplier,DataType
0,C&L 01,2010-11,5.1386,233685362.4,45459818.0,Ascending,NaN,1000.0,1.0,Real_Annual
1,C&L 01,2011-12,4.5777,220407850.9,48202343.0,Ascending,NaN,1000.0,1.0,Real_Annual
2,C&L 01,2012-13,4.3743,225690062.0,51624697.0,Ascending,NaN,1000.0,1.0,Real_Annual


,Code,LocalAuthority,Period,Value,Numerator,Denominator,FG_Type,Ranking_Type,Ranking_GoldilocksMidpoint,Family_Group,DataType
0,C&L 01,Aberdeen City,2010-11,0.4777,922049.9,1922292.0,"Environmental, Culture & Leisure, Economic Dev...",Ascending,NaN,Family Group 4,Real_Annual
1,C&L 01,Aberdeen City,2011-12,1.0771,2202377.8,2045051.0,"Environmental, Culture & Leisure, Economic Dev...",Ascending,NaN,Family Group 4,Real_Annual
2,C&L 01,Aberdeen City,2012-13,5.0708,10981647.6,2163756.0,"Environmental, Culture & Leisure, Economic Dev...",Ascending,NaN,Family Group 4,Real_Annual


## Calculate Averages

### Create DataFrames for Average Calculations

In [13]:
data_scotaveragecleaned = data[[
                                'Code',
                                'Period',
                                'Value',
                                'Numerator',
                                'Denominator',
                                'DataType'
                            ]]
data_fgaveragecleaned = data[[
                                'Code',
                                'Period',
                                'Family_Group',
                                'Value',
                                'Numerator',
                                'Denominator',
                                'DataType'
                            ]]

### Mean and Medians - 32 Council (ScotAverages)

In [14]:
scotAverages_Mean = data_scotaveragecleaned.groupby(['Code', 'Period','DataType'], as_index=False).mean()
scotAverages_Median = data_scotaveragecleaned.groupby(['Code', 'Period','DataType'], as_index=False).median()

displaywithHeader(scotAverages_Mean,'scotAverages_Mean')
displaywithHeader(scotAverages_Median,'scotAverages_Median')

,Code,Period,DataType,Value,Numerator,Denominator
0,C&L 01,2010-11,Cash_Annual,3.297813,5.045062e+06,1.420619e+06
1,C&L 01,2010-11,Real_Annual,4.773550,7.302668e+06,1.420619e+06
2,C&L 01,2011-12,Cash_Annual,3.134688,4.860000e+06,1.506323e+06


,Code,Period,DataType,Value,Numerator,Denominator
0,C&L 01,2010-11,Cash_Annual,3.44000,3626500.0,1080558.5
1,C&L 01,2010-11,Real_Annual,4.97935,5249315.3,1080558.5
2,C&L 01,2011-12,Cash_Annual,2.89000,3356500.0,1099975.5


### Sum Num/Den - 32 Council (ScotAverages)

In [15]:
scotAverages_NumDenSums = data_scotaveragecleaned.groupby(['Code', 'Period', 'DataType'], as_index=False).sum()
scotAverages_NumDenSums['Scot_NumDenAv'] = scotAverages_NumDenSums['Numerator'] / scotAverages_NumDenSums['Denominator']
scotAverages_NumDenSums = scotAverages_NumDenSums[['Code', 'Period', 'DataType', 'Scot_NumDenAv']]
scotAverages_NumDenSums = scotAverages_NumDenSums.replace([np.inf, -np.inf], np.nan)

displaywithHeader(scotAverages_NumDenSums,'scotAverages_NumDenSums')

,Code,Period,DataType,Scot_NumDenAv
0,C&L 01,2010-11,Cash_Annual,3.551312
1,C&L 01,2010-11,Real_Annual,5.140482
2,C&L 01,2011-12,Cash_Annual,3.226399


### Merge, rename Columns Add Keys and Sort - 32 Council (ScotAverages)

In [16]:
# Merge DataFrames
scotAverages = scotAverages_Mean.merge(scotAverages_Median, how='left', on=['Code', 'Period','DataType'], suffixes=('_Mean', '_Median'))
scotAverages = scotAverages.merge(scotAverages_NumDenSums, how='left', on=['Code', 'Period', 'DataType'])

# Add Keys
scotAverages['Key_CodePeriodDType'] = scotAverages['Code'] + scotAverages['Period'] + scotAverages['DataType']
scotAverages = scotAverages[['Key_CodePeriodDType','Code', 'Period','DataType', 'Value_Mean', 'Numerator_Mean', 'Denominator_Mean', 'Value_Median', 'Numerator_Median', 'Denominator_Median', 'Scot_NumDenAv']]

# Rename Columns
scotAverages = scotAverages.rename(columns={
    'Value_Mean': 'Value_Scot_Mean',
    'Numerator_Mean': 'Numerator_Scot_Mean',
    'Denominator_Mean': 'Denominator_Scot_Mean',
    'Value_Median': 'Value_Scot_Median',
    'Numerator_Median': 'Numerator_Scot_Median',
    'Denominator_Median': 'Denominator_Scot_Median'
})

displaywithHeader(scotAverages,'scotAverages')

,Key_CodePeriodDType,Code,Period,DataType,Value_Scot_Mean,Numerator_Scot_Mean,Denominator_Scot_Mean,Value_Scot_Median,Numerator_Scot_Median,Denominator_Scot_Median,Scot_NumDenAv
0,C&L 012010-11Cash_Annual,C&L 01,2010-11,Cash_Annual,3.297813,5.045062e+06,1.420619e+06,3.44000,3626500.0,1080558.5,3.551312
1,C&L 012010-11Real_Annual,C&L 01,2010-11,Real_Annual,4.773550,7.302668e+06,1.420619e+06,4.97935,5249315.3,1080558.5,5.140482
2,C&L 012011-12Cash_Annual,C&L 01,2011-12,Cash_Annual,3.134688,4.860000e+06,1.506323e+06,2.89000,3356500.0,1099975.5,3.226399


### Mean and Medians - Family Groups

In [17]:
FGAverages_Mean = data_fgaveragecleaned.groupby(['Code', 'Family_Group', 'Period', 'DataType'], as_index=False).mean()
FGAverages_Median = data_fgaveragecleaned.groupby(['Code', 'Family_Group', 'Period', 'DataType'], as_index=False).median()

displaywithHeader(FGAverages_Mean,'FGAverages_Mean')
displaywithHeader(FGAverages_Median,'FGAverages_Median')

,Code,Family_Group,Period,DataType,Value,Numerator,Denominator
0,C&L 01,Family Group 1,2010-11,Cash_Annual,3.01125,3.076125e+06,923124.125
1,C&L 01,Family Group 1,2010-11,Real_Annual,4.35875,4.452654e+06,923124.125
2,C&L 01,Family Group 1,2011-12,Cash_Annual,3.03875,3.213375e+06,1036476.000


,Code,Family_Group,Period,DataType,Value,Numerator,Denominator
0,C&L 01,Family Group 1,2010-11,Cash_Annual,3.2450,1955000.0,695444.0
1,C&L 01,Family Group 1,2010-11,Real_Annual,4.6971,2829839.1,695444.0
2,C&L 01,Family Group 1,2011-12,Cash_Annual,2.8650,1825500.0,707515.5


### Sum Num/Den - Family Groups

In [18]:
FGAverages_NumDenSums = data_fgaveragecleaned.groupby(['Code', 'Family_Group', 'Period', 'DataType'], as_index=False).sum()
FGAverages_NumDenSums['FG_NumDenAv'] = FGAverages_NumDenSums['Numerator'] / FGAverages_NumDenSums['Denominator']
FGAverages_NumDenSums = FGAverages_NumDenSums[['Code', 'Family_Group', 'Period','DataType', 'FG_NumDenAv']]
FGAverages_NumDenSums = FGAverages_NumDenSums.replace([np.inf, -np.inf], np.nan)

displaywithHeader(FGAverages_NumDenSums,'FGAverages_NumDenSums')

,Code,Family_Group,Period,DataType,FG_NumDenAv
0,C&L 01,Family Group 1,2010-11,Cash_Annual,3.332298
1,C&L 01,Family Group 1,2010-11,Real_Annual,4.823462
2,C&L 01,Family Group 1,2011-12,Cash_Annual,3.100289


### Merge, rename Columns Add Keys and Sort - Family Groups

In [19]:
# Merge DataFrames
FGAverages = FGAverages_Mean.merge(FGAverages_Median, how='left', on=['Code','Family_Group','Period', 'DataType'], suffixes=('_Mean', '_Median'))
FGAverages = FGAverages.merge(FGAverages_NumDenSums, how='left', on=['Code', 'Family_Group', 'Period', 'DataType'])

# Add Keys
FGAverages['Key_CodePeriodDType'] = FGAverages['Code'] + FGAverages['Period'] + FGAverages['DataType'] 
FGAverages = FGAverages[['Key_CodePeriodDType', 'Code', 'Period', 'DataType', 'Family_Group', 'Value_Mean', 'Numerator_Mean', 'Denominator_Mean', 'Value_Median', 'Numerator_Median', 'Denominator_Median', 'FG_NumDenAv']]

# Rename Columns
FGAverages = FGAverages.rename(columns = {
    'Value_Mean': 'Value_FG_Mean',
    'Numerator_Mean': 'Numerator_FG_Mean',
    'Denominator_Mean': 'Denominator_FG_Mean',
    'Value_Median': 'Value_FG_Median',
    'Numerator_Median': 'Numerator_FG_Median',
    'Denominator_Median': 'Denominator_FG_Median'
})

displaywithHeader(FGAverages,'FGAverages')

,Key_CodePeriodDType,Code,Period,DataType,Family_Group,Value_FG_Mean,Numerator_FG_Mean,Denominator_FG_Mean,Value_FG_Median,Numerator_FG_Median,Denominator_FG_Median,FG_NumDenAv
0,C&L 012010-11Cash_Annual,C&L 01,2010-11,Cash_Annual,Family Group 1,3.01125,3.076125e+06,923124.125,3.2450,1955000.0,695444.0,3.332298
1,C&L 012010-11Real_Annual,C&L 01,2010-11,Real_Annual,Family Group 1,4.35875,4.452654e+06,923124.125,4.6971,2829839.1,695444.0,4.823462
2,C&L 012011-12Cash_Annual,C&L 01,2011-12,Cash_Annual,Family Group 1,3.03875,3.213375e+06,1036476.000,2.8650,1825500.0,707515.5,3.100289


## Calculate Rankings

### Create Dataframes for Ranking Calculations

In [20]:
data_fgrankscleaned = data[[
    'Code',
    'LocalAuthority',
    'Period',
    'Value',
    'Family_Group',
    'Ranking_GoldilocksMidpoint',
    'DataType'
]]

familyRanks = data[[
    'Code',
    'LocalAuthority',
    'Period',
    'DataType'
]]

data_scotrankscleaned = data[[
                'Code',
                'LocalAuthority',
                'Period',
                'Value',
                'Ranking_GoldilocksMidpoint',
                'DataType'
            ]]

scotRanks = data[[
    'Code',
    'LocalAuthority',
    'Period',
    'DataType'
]]

# Avoids Potential Writeback Issues
familyRanks = familyRanks.copy(deep=True)
familyRanksGoldi = data_fgrankscleaned.copy(deep=True)

scotRanks = scotRanks.copy(deep=True)
scotRanksGoldi = data_scotrankscleaned.copy(deep=True)

### Calculate Ascending and Descending Rankings

In [21]:
familyRanks['FamilyRank_Desc'] = data_fgrankscleaned.groupby(['Code', 'Period', 'Family_Group', 'DataType'])['Value'].rank('min', ascending=False).astype(int)
familyRanks['FamilyRank_Asc'] = data_fgrankscleaned.groupby(['Code', 'Period', 'Family_Group', 'DataType'])['Value'].rank('min', ascending=True).astype(int)
familyRanks['FamilyRank_Desc_Pct'] = data_fgrankscleaned.groupby(['Code', 'Period', 'Family_Group', 'DataType'])['Value'].rank('min', ascending=False, pct=True).astype(float)
familyRanks['FamilyRank_Asc_Pct'] = data_fgrankscleaned.groupby(['Code', 'Period', 'Family_Group', 'DataType'])['Value'].rank('min', ascending=True, pct=True).astype(float)

scotRanks['ScotRank_Desc'] = data_scotrankscleaned.groupby(['Code', 'Period', 'DataType'])['Value'].rank('min', ascending=False).astype(int)
scotRanks['ScotRank_Asc'] = data_scotrankscleaned.groupby(['Code', 'Period', 'DataType'])['Value'].rank('min', ascending=True).astype(int)
scotRanks['ScotRank_Desc_Pct'] = data_scotrankscleaned.groupby(['Code', 'Period','DataType'])['Value'].rank('min', ascending=False, pct=True).astype(float)
scotRanks['ScotRank_Asc_Pct'] = data_scotrankscleaned.groupby(['Code', 'Period', 'DataType'])['Value'].rank('min', ascending=True, pct=True).astype(float)

displaywithHeader(familyRanks,'familyRanks')
displaywithHeader(scotRanks,'scotRanks')

,Code,LocalAuthority,Period,DataType,FamilyRank_Desc,FamilyRank_Asc,FamilyRank_Desc_Pct,FamilyRank_Asc_Pct
0,C&L 01,Aberdeen City,2010-11,Real_Annual,8,1,1.0,0.125
1,C&L 01,Aberdeen City,2011-12,Real_Annual,8,1,1.0,0.125
2,C&L 01,Aberdeen City,2012-13,Real_Annual,4,5,0.5,0.625


,Code,LocalAuthority,Period,DataType,ScotRank_Desc,ScotRank_Asc,ScotRank_Desc_Pct,ScotRank_Asc_Pct
0,C&L 01,Aberdeen City,2010-11,Real_Annual,32,1,1.00000,0.03125
1,C&L 01,Aberdeen City,2011-12,Real_Annual,32,1,1.00000,0.03125
2,C&L 01,Aberdeen City,2012-13,Real_Annual,9,24,0.28125,0.75000


### Calculate FG Goldilocks Ranking

In [22]:
# Define functions to determine distance between two values and use this to return absolute distance from goldilocks value
def distance(Current, Previous):
    return (max(Previous, Current) - min(Previous, Current)) * (-1 if Previous > Current else 1)

def DifferenceFromGoldilocksMidPoint(df):
    if df['Ranking_GoldilocksMidpoint'] == None:
        return None
    else:
        return abs(distance(df['Value'], df['Ranking_GoldilocksMidpoint']))

# Apply Functions Above to Calculate Goldilocks Rankings
familyRanksGoldi['AbsoluteDifferenceFromGoldilocksMidPoint'] = familyRanksGoldi.apply(DifferenceFromGoldilocksMidPoint, axis=1)
familyRanksGoldi = familyRanksGoldi[pd.notnull(familyRanksGoldi['AbsoluteDifferenceFromGoldilocksMidPoint'])]
familyRanksGoldi['FamilyRank_Goldi'] = familyRanksGoldi.groupby(['Code', 'Period', 'Family_Group', 'DataType'])['AbsoluteDifferenceFromGoldilocksMidPoint'].rank('min', ascending=True).astype(int)
familyRanksGoldi['FamilyRank_Goldi_Pct'] = familyRanksGoldi.groupby(['Code', 'Period', 'Family_Group', 'DataType'])['AbsoluteDifferenceFromGoldilocksMidPoint'].rank('min', ascending=True, pct=True).astype(float)

scotRanksGoldi['AbsoluteDifferenceFromGoldilocksMidPoint'] = scotRanksGoldi.apply(DifferenceFromGoldilocksMidPoint, axis=1)
scotRanksGoldi = scotRanksGoldi[pd.notnull(scotRanksGoldi['AbsoluteDifferenceFromGoldilocksMidPoint'])]
scotRanksGoldi['ScotRank_Goldi'] = scotRanksGoldi.groupby(['Code', 'Period', 'DataType'])['AbsoluteDifferenceFromGoldilocksMidPoint'].rank('min', ascending=True).astype(int)
scotRanksGoldi['ScotRank_Goldi_Pct'] = scotRanksGoldi.groupby(['Code', 'Period', 'DataType'])['AbsoluteDifferenceFromGoldilocksMidPoint'].rank('min', ascending=True, pct=True).astype(float)

displaywithHeader(familyRanksGoldi,'familyRanksGoldi')
displaywithHeader(scotRanksGoldi,'scotRanksGoldi')

,Code,LocalAuthority,Period,Value,Family_Group,Ranking_GoldilocksMidpoint,DataType,AbsoluteDifferenceFromGoldilocksMidPoint,FamilyRank_Goldi,FamilyRank_Goldi_Pct
17586,CORP 03b,Aberdeen City,2010-11,0.4665,Family Group 4,0.5,Real_Annual,0.0335,4,0.500
17587,CORP 03b,Aberdeen City,2011-12,0.4491,Family Group 4,0.5,Real_Annual,0.0509,6,0.750
17588,CORP 03b,Aberdeen City,2012-13,0.4969,Family Group 4,0.5,Real_Annual,0.0031,1,0.125


,Code,LocalAuthority,Period,Value,Ranking_GoldilocksMidpoint,DataType,AbsoluteDifferenceFromGoldilocksMidPoint,ScotRank_Goldi,ScotRank_Goldi_Pct
17586,CORP 03b,Aberdeen City,2010-11,0.4665,0.5,Real_Annual,0.0335,11,0.34375
17587,CORP 03b,Aberdeen City,2011-12,0.4491,0.5,Real_Annual,0.0509,20,0.62500
17588,CORP 03b,Aberdeen City,2012-13,0.4969,0.5,Real_Annual,0.0031,1,0.03125


### Merge Ranking Tables

In [23]:
familyRanks = familyRanks.merge(familyRanksGoldi[['Code', 'Period', 'LocalAuthority', 'FamilyRank_Goldi', 'FamilyRank_Goldi_Pct', 'DataType']], how='left', on=['Code', 'Period', 'LocalAuthority', 'DataType'], suffixes=('_FamilyRank', '_Goldi'))
familyRanks = familyRanks.merge(info, how = 'left', left_on= 'Code', right_on = 'Code_Sortable')

scotRanks = scotRanks.merge(scotRanksGoldi[['Code', 'Period', 'LocalAuthority', 'ScotRank_Goldi', 'ScotRank_Goldi_Pct', 'DataType']], how='left', on=['Code', 'Period', 'LocalAuthority', 'DataType'], suffixes=('_ScotRank', '_Goldi'))
scotRanks = scotRanks.merge(info, how='left', left_on='Code', right_on='Code_Sortable')

displaywithHeader(familyRanks,'familyRanks')
displaywithHeader(scotRanks,'scotRanks')

,Code_x,LocalAuthority,Period,DataType,FamilyRank_Desc,FamilyRank_Asc,FamilyRank_Desc_Pct,FamilyRank_Asc_Pct,FamilyRank_Goldi,FamilyRank_Goldi_Pct,Title,Code_y,Code_Sortable,ReportingPeriod,MeasureType,NumberFormat,YMin,YMax,ISCategory,Committee,FamilyGrouping,StirlingService,Ranking_Type,NumberFormat_NoText,Source,Numerator_Correct,Denominator_Correct,Numerator_Match,Denominator_Match,Numerator_Multipier,Denominator_Multiplier,Ranking_GoldilocksMidpoint,NumberFormat_Axis,Format_Python,FormatAxis_Python,AdditionalAxisDenominator_Python,FormatAxis_Plotly_Prefix,FormatAxis_Plotly_Suffix,ImgPxlWidth_Plotly,ImgPxlHeight_Plotly,SubGroup_PythonReport,YMin_Plotly,YMax_Plotly,OData__ColorTag,Group_PythonReport
0,C&L 01,Aberdeen City,2010-11,Real_Annual,8,1,1.0,0.125,NaN,NaN,Cost per Attendance at Sports Facilities,C&L01,C&L 01,Annual,Cost,'£ '0.00,0.0,NaN,Culture & Leisure Services,Community Planning and Regeneration,"Environmental, Culture & Leisure, Economic Dev...",Economic Development & Communities ; Economic ...,Ascending,0,NaN,Sports facilities including swimming pools - n...,No. Of Attendances,Sports facilities including swimming pools - n...,No. Of Attendances,1000.0,1.0,NaN,'£ '0.00,£ {:0.2f},",d",1,£,NaN,790,305,Leisure Facilities & Attractions,NaN,NaN,NaN,Economy
1,C&L 01,Aberdeen City,2011-12,Real_Annual,8,1,1.0,0.125,NaN,NaN,Cost per Attendance at Sports Facilities,C&L01,C&L 01,Annual,Cost,'£ '0.00,0.0,NaN,Culture & Leisure Services,Community Planning and Regeneration,"Environmental, Culture & Leisure, Economic Dev...",Economic Development & Communities ; Economic ...,Ascending,0,NaN,Sports facilities including swimming pools - n...,No. Of Attendances,Sports facilities including swimming pools - n...,No. Of Attendances,1000.0,1.0,NaN,'£ '0.00,£ {:0.2f},",d",1,£,NaN,790,305,Leisure Facilities & Attractions,NaN,NaN,NaN,Economy
2,C&L 01,Aberdeen City,2012-13,Real_Annual,4,5,0.5,0.625,NaN,NaN,Cost per Attendance at Sports Facilities,C&L01,C&L 01,Annual,Cost,'£ '0.00,0.0,NaN,Culture & Leisure Services,Community Planning and Regeneration,"Environmental, Culture & Leisure, Economic Dev...",Economic Development & Communities ; Economic ...,Ascending,0,NaN,Sports facilities including swimming pools - n...,No. Of Attendances,Sports facilities including swimming pools - n...,No. Of Attendances,1000.0,1.0,NaN,'£ '0.00,£ {:0.2f},",d",1,£,NaN,790,305,Leisure Facilities & Attractions,NaN,NaN,NaN,Economy


,Code_x,LocalAuthority,Period,DataType,ScotRank_Desc,ScotRank_Asc,ScotRank_Desc_Pct,ScotRank_Asc_Pct,ScotRank_Goldi,ScotRank_Goldi_Pct,Title,Code_y,Code_Sortable,ReportingPeriod,MeasureType,NumberFormat,YMin,YMax,ISCategory,Committee,FamilyGrouping,StirlingService,Ranking_Type,NumberFormat_NoText,Source,Numerator_Correct,Denominator_Correct,Numerator_Match,Denominator_Match,Numerator_Multipier,Denominator_Multiplier,Ranking_GoldilocksMidpoint,NumberFormat_Axis,Format_Python,FormatAxis_Python,AdditionalAxisDenominator_Python,FormatAxis_Plotly_Prefix,FormatAxis_Plotly_Suffix,ImgPxlWidth_Plotly,ImgPxlHeight_Plotly,SubGroup_PythonReport,YMin_Plotly,YMax_Plotly,OData__ColorTag,Group_PythonReport
0,C&L 01,Aberdeen City,2010-11,Real_Annual,32,1,1.00000,0.03125,NaN,NaN,Cost per Attendance at Sports Facilities,C&L01,C&L 01,Annual,Cost,'£ '0.00,0.0,NaN,Culture & Leisure Services,Community Planning and Regeneration,"Environmental, Culture & Leisure, Economic Dev...",Economic Development & Communities ; Economic ...,Ascending,0,NaN,Sports facilities including swimming pools - n...,No. Of Attendances,Sports facilities including swimming pools - n...,No. Of Attendances,1000.0,1.0,NaN,'£ '0.00,£ {:0.2f},",d",1,£,NaN,790,305,Leisure Facilities & Attractions,NaN,NaN,NaN,Economy
1,C&L 01,Aberdeen City,2011-12,Real_Annual,32,1,1.00000,0.03125,NaN,NaN,Cost per Attendance at Sports Facilities,C&L01,C&L 01,Annual,Cost,'£ '0.00,0.0,NaN,Culture & Leisure Services,Community Planning and Regeneration,"Environmental, Culture & Leisure, Economic Dev...",Economic Development & Communities ; Economic ...,Ascending,0,NaN,Sports facilities including swimming pools - n...,No. Of Attendances,Sports facilities including swimming pools - n...,No. Of Attendances,1000.0,1.0,NaN,'£ '0.00,£ {:0.2f},",d",1,£,NaN,790,305,Leisure Facilities & Attractions,NaN,NaN,NaN,Economy
2,C&L 01,Aberdeen City,2012-13,Real_Annual,9,24,0.28125,0.75000,NaN,NaN,Cost per Attendance at Sports Facilities,C&L01,C&L 01,Annual,Cost,'£ '0.00,0.0,NaN,Culture & Leisure Services,Community Planning and Regeneration,"Environmental, Culture & Leisure, Economic Dev...",Economic Development & Communities ; Economic ...,Ascending,0,NaN,Sports facilities including swimming pools - n...,No. Of Attendances,Sports facilities including swimming pools - n...,No. Of Attendances,1000.0,1.0,NaN,'£ '0.00,£ {:0.2f},",d",1,£,NaN,790,305,Leisure Facilities & Attractions,NaN,NaN,NaN,Economy


### Select Correct Ranking

In [24]:
# Define functions needed to select correct ranking type and percentile type
def Rank_select(df, dftitle):
    if df['Ranking_Type'] == "Ascending" and dftitle == 'familyRanks':
        return df['FamilyRank_Asc']
    elif df['Ranking_Type'] == "Ascending" and dftitle == 'scotRanks':
        return df['ScotRank_Asc']
    elif df['Ranking_Type'] == "Descending" and dftitle == 'familyRanks':
        return df['FamilyRank_Desc']
    elif df['Ranking_Type'] == "Descending" and dftitle == 'scotRanks':
        return df['ScotRank_Desc']
    elif df['Ranking_Type'] == "Goldilocks" and dftitle == 'familyRanks':
        return df['FamilyRank_Goldi']
    elif df['Ranking_Type'] == "Goldilocks" and dftitle == 'scotRanks':
        return df['ScotRank_Goldi']
    else:
        return None

def Rank_Pct_select(df, dftitle):
    if df['Ranking_Type'] == "Ascending" and dftitle == 'familyRanks':
        return df['FamilyRank_Asc_Pct']
    if df['Ranking_Type'] == "Ascending" and dftitle == 'scotRanks':
        return df['ScotRank_Asc_Pct']
    elif df['Ranking_Type'] == "Descending" and dftitle == 'familyRanks':
        return df['FamilyRank_Desc_Pct']
    elif df['Ranking_Type'] == "Descending" and dftitle == 'scotRanks':
        return df['ScotRank_Desc_Pct']
    elif df['Ranking_Type'] == "Goldilocks" and dftitle == 'familyRanks':
        return df['FamilyRank_Goldi_Pct']
    elif df['Ranking_Type'] == "Goldilocks" and dftitle == 'scotRanks':
        return df['ScotRank_Goldi_Pct']
    else:
        return None

# Apply functions above to create two new columns that contain the correct rank and percentile for each row
familyRanks['FamilyRank'] = familyRanks.apply(Rank_select, axis=1, dftitle = 'familyRanks')
familyRanks['FamilyPct'] = familyRanks.apply(Rank_Pct_select, axis=1, dftitle = 'familyRanks')

# Apply functions above to create two new columns that contain the correct rank and percentile for each row
scotRanks['ScotRank'] = scotRanks.apply(Rank_select, axis=1, dftitle = 'scotRanks')
scotRanks['ScotPct'] = scotRanks.apply(Rank_Pct_select, axis=1, dftitle = 'scotRanks')

displaywithHeader(familyRanks,'familyRanks')
displaywithHeader(familyRanks,'scotRanks')

,Code_x,LocalAuthority,Period,DataType,FamilyRank_Desc,FamilyRank_Asc,FamilyRank_Desc_Pct,FamilyRank_Asc_Pct,FamilyRank_Goldi,FamilyRank_Goldi_Pct,Title,Code_y,Code_Sortable,ReportingPeriod,MeasureType,NumberFormat,YMin,YMax,ISCategory,Committee,FamilyGrouping,StirlingService,Ranking_Type,NumberFormat_NoText,Source,Numerator_Correct,Denominator_Correct,Numerator_Match,Denominator_Match,Numerator_Multipier,Denominator_Multiplier,Ranking_GoldilocksMidpoint,NumberFormat_Axis,Format_Python,FormatAxis_Python,AdditionalAxisDenominator_Python,FormatAxis_Plotly_Prefix,FormatAxis_Plotly_Suffix,ImgPxlWidth_Plotly,ImgPxlHeight_Plotly,SubGroup_PythonReport,YMin_Plotly,YMax_Plotly,OData__ColorTag,Group_PythonReport,FamilyRank,FamilyPct
0,C&L 01,Aberdeen City,2010-11,Real_Annual,8,1,1.0,0.125,NaN,NaN,Cost per Attendance at Sports Facilities,C&L01,C&L 01,Annual,Cost,'£ '0.00,0.0,NaN,Culture & Leisure Services,Community Planning and Regeneration,"Environmental, Culture & Leisure, Economic Dev...",Economic Development & Communities ; Economic ...,Ascending,0,NaN,Sports facilities including swimming pools - n...,No. Of Attendances,Sports facilities including swimming pools - n...,No. Of Attendances,1000.0,1.0,NaN,'£ '0.00,£ {:0.2f},",d",1,£,NaN,790,305,Leisure Facilities & Attractions,NaN,NaN,NaN,Economy,1.0,0.125
1,C&L 01,Aberdeen City,2011-12,Real_Annual,8,1,1.0,0.125,NaN,NaN,Cost per Attendance at Sports Facilities,C&L01,C&L 01,Annual,Cost,'£ '0.00,0.0,NaN,Culture & Leisure Services,Community Planning and Regeneration,"Environmental, Culture & Leisure, Economic Dev...",Economic Development & Communities ; Economic ...,Ascending,0,NaN,Sports facilities including swimming pools - n...,No. Of Attendances,Sports facilities including swimming pools - n...,No. Of Attendances,1000.0,1.0,NaN,'£ '0.00,£ {:0.2f},",d",1,£,NaN,790,305,Leisure Facilities & Attractions,NaN,NaN,NaN,Economy,1.0,0.125
2,C&L 01,Aberdeen City,2012-13,Real_Annual,4,5,0.5,0.625,NaN,NaN,Cost per Attendance at Sports Facilities,C&L01,C&L 01,Annual,Cost,'£ '0.00,0.0,NaN,Culture & Leisure Services,Community Planning and Regeneration,"Environmental, Culture & Leisure, Economic Dev...",Economic Development & Communities ; Economic ...,Ascending,0,NaN,Sports facilities including swimming pools - n...,No. Of Attendances,Sports facilities including swimming pools - n...,No. Of Attendances,1000.0,1.0,NaN,'£ '0.00,£ {:0.2f},",d",1,£,NaN,790,305,Leisure Facilities & Attractions,NaN,NaN,NaN,Economy,5.0,0.625


,Code_x,LocalAuthority,Period,DataType,FamilyRank_Desc,FamilyRank_Asc,FamilyRank_Desc_Pct,FamilyRank_Asc_Pct,FamilyRank_Goldi,FamilyRank_Goldi_Pct,Title,Code_y,Code_Sortable,ReportingPeriod,MeasureType,NumberFormat,YMin,YMax,ISCategory,Committee,FamilyGrouping,StirlingService,Ranking_Type,NumberFormat_NoText,Source,Numerator_Correct,Denominator_Correct,Numerator_Match,Denominator_Match,Numerator_Multipier,Denominator_Multiplier,Ranking_GoldilocksMidpoint,NumberFormat_Axis,Format_Python,FormatAxis_Python,AdditionalAxisDenominator_Python,FormatAxis_Plotly_Prefix,FormatAxis_Plotly_Suffix,ImgPxlWidth_Plotly,ImgPxlHeight_Plotly,SubGroup_PythonReport,YMin_Plotly,YMax_Plotly,OData__ColorTag,Group_PythonReport,FamilyRank,FamilyPct
0,C&L 01,Aberdeen City,2010-11,Real_Annual,8,1,1.0,0.125,NaN,NaN,Cost per Attendance at Sports Facilities,C&L01,C&L 01,Annual,Cost,'£ '0.00,0.0,NaN,Culture & Leisure Services,Community Planning and Regeneration,"Environmental, Culture & Leisure, Economic Dev...",Economic Development & Communities ; Economic ...,Ascending,0,NaN,Sports facilities including swimming pools - n...,No. Of Attendances,Sports facilities including swimming pools - n...,No. Of Attendances,1000.0,1.0,NaN,'£ '0.00,£ {:0.2f},",d",1,£,NaN,790,305,Leisure Facilities & Attractions,NaN,NaN,NaN,Economy,1.0,0.125
1,C&L 01,Aberdeen City,2011-12,Real_Annual,8,1,1.0,0.125,NaN,NaN,Cost per Attendance at Sports Facilities,C&L01,C&L 01,Annual,Cost,'£ '0.00,0.0,NaN,Culture & Leisure Services,Community Planning and Regeneration,"Environmental, Culture & Leisure, Economic Dev...",Economic Development & Communities ; Economic ...,Ascending,0,NaN,Sports facilities including swimming pools - n...,No. Of Attendances,Sports facilities including swimming pools - n...,No. Of Attendances,1000.0,1.0,NaN,'£ '0.00,£ {:0.2f},",d",1,£,NaN,790,305,Leisure Facilities & Attractions,NaN,NaN,NaN,Economy,1.0,0.125
2,C&L 01,Aberdeen City,2012-13,Real_Annual,4,5,0.5,0.625,NaN,NaN,Cost per Attendance at Sports Facilities,C&L01,C&L 01,Annual,Cost,'£ '0.00,0.0,NaN,Culture & Leisure Services,Community Planning and Regeneration,"Environmental, Culture & Leisure, Economic Dev...",Economic Development & Communities ; Economic ...,Ascending,0,NaN,Sports facilities including swimming pools - n...,No. Of Attendances,Sports facilities including swimming pools - n...,No. Of Attendances,1000.0,1.0,NaN,'£ '0.00,£ {:0.2f},",d",1,£,NaN,790,305,Leisure Facilities & Attractions,NaN,NaN,NaN,Economy,5.0,0.625


### Select and Rename Columns, Merge into 'data' Dataframe

In [25]:
familyRanks = familyRanks[[
                'Code_Sortable',
                'LocalAuthority',
                'Period',
                'FamilyRank',
                'FamilyPct',
                'DataType'
            ]]

scotRanks = scotRanks[[
                'Code_Sortable',
                'LocalAuthority',
                'Period',
                'ScotRank',
                'ScotPct',
                'DataType'
            ]]

ranks = familyRanks.merge(scotRanks, how='left', on = ['Code_Sortable','LocalAuthority','Period','DataType'])
ranks = ranks.rename(columns = {'Code_Sortable': 'Code'})
data = data.merge(ranks,how = 'left', on = ['Code','LocalAuthority','Period','DataType'])

displaywithHeader(data,'data')

,Code,LocalAuthority,Period,Value,Numerator,Denominator,FG_Type,Ranking_Type,Ranking_GoldilocksMidpoint,Family_Group,DataType,FamilyRank,FamilyPct,ScotRank,ScotPct
0,C&L 01,Aberdeen City,2010-11,0.4777,922049.9,1922292.0,"Environmental, Culture & Leisure, Economic Dev...",Ascending,NaN,Family Group 4,Real_Annual,1.0,0.125,1.0,0.03125
1,C&L 01,Aberdeen City,2011-12,1.0771,2202377.8,2045051.0,"Environmental, Culture & Leisure, Economic Dev...",Ascending,NaN,Family Group 4,Real_Annual,1.0,0.125,1.0,0.03125
2,C&L 01,Aberdeen City,2012-13,5.0708,10981647.6,2163756.0,"Environmental, Culture & Leisure, Economic Dev...",Ascending,NaN,Family Group 4,Real_Annual,5.0,0.625,24.0,0.75000


## Clean 'data' df

### Add Keys

In [26]:
data['Key_CodePeriodDType'] = data['Code'] + data['Period'] + data['DataType']
data['Key_CodePeriodFamilyGroupDType'] = data['Code'] + data['Period'] + data['Family_Group'] + data['DataType']
data['Key_CodePeriodLADType'] = data['Code'] + data['Period'] + data['LocalAuthority'] + data['DataType']
data['Iter_CodeLADType'] = data['Code'] + data['LocalAuthority'] + data['DataType']

displaywithHeader(data,'data')

,Code,LocalAuthority,Period,Value,Numerator,Denominator,FG_Type,Ranking_Type,Ranking_GoldilocksMidpoint,Family_Group,DataType,FamilyRank,FamilyPct,ScotRank,ScotPct,Key_CodePeriodDType,Key_CodePeriodFamilyGroupDType,Key_CodePeriodLADType,Iter_CodeLADType
0,C&L 01,Aberdeen City,2010-11,0.4777,922049.9,1922292.0,"Environmental, Culture & Leisure, Economic Dev...",Ascending,NaN,Family Group 4,Real_Annual,1.0,0.125,1.0,0.03125,C&L 012010-11Real_Annual,C&L 012010-11Family Group 4Real_Annual,C&L 012010-11Aberdeen CityReal_Annual,C&L 01Aberdeen CityReal_Annual
1,C&L 01,Aberdeen City,2011-12,1.0771,2202377.8,2045051.0,"Environmental, Culture & Leisure, Economic Dev...",Ascending,NaN,Family Group 4,Real_Annual,1.0,0.125,1.0,0.03125,C&L 012011-12Real_Annual,C&L 012011-12Family Group 4Real_Annual,C&L 012011-12Aberdeen CityReal_Annual,C&L 01Aberdeen CityReal_Annual
2,C&L 01,Aberdeen City,2012-13,5.0708,10981647.6,2163756.0,"Environmental, Culture & Leisure, Economic Dev...",Ascending,NaN,Family Group 4,Real_Annual,5.0,0.625,24.0,0.75000,C&L 012012-13Real_Annual,C&L 012012-13Family Group 4Real_Annual,C&L 012012-13Aberdeen CityReal_Annual,C&L 01Aberdeen CityReal_Annual


### Select Only Required Columns

In [27]:
data = data[
                [
                    'Key_CodePeriodDType',
                    'Key_CodePeriodFamilyGroupDType',
                    'Key_CodePeriodLADType',
                    'Code',
                    'LocalAuthority',
                    'Period',
                    'Value',
                    'Numerator',
                    'Denominator',
                    'FamilyRank',
                    'FamilyPct',
                    'ScotRank',
                    'ScotPct',
                    'DataType',
                    'Iter_CodeLADType'
                ]
            ]

displaywithHeader(data,'data')

,Key_CodePeriodDType,Key_CodePeriodFamilyGroupDType,Key_CodePeriodLADType,Code,LocalAuthority,Period,Value,Numerator,Denominator,FamilyRank,FamilyPct,ScotRank,ScotPct,DataType,Iter_CodeLADType
0,C&L 012010-11Real_Annual,C&L 012010-11Family Group 4Real_Annual,C&L 012010-11Aberdeen CityReal_Annual,C&L 01,Aberdeen City,2010-11,0.4777,922049.9,1922292.0,1.0,0.125,1.0,0.03125,Real_Annual,C&L 01Aberdeen CityReal_Annual
1,C&L 012011-12Real_Annual,C&L 012011-12Family Group 4Real_Annual,C&L 012011-12Aberdeen CityReal_Annual,C&L 01,Aberdeen City,2011-12,1.0771,2202377.8,2045051.0,1.0,0.125,1.0,0.03125,Real_Annual,C&L 01Aberdeen CityReal_Annual
2,C&L 012012-13Real_Annual,C&L 012012-13Family Group 4Real_Annual,C&L 012012-13Aberdeen CityReal_Annual,C&L 01,Aberdeen City,2012-13,5.0708,10981647.6,2163756.0,5.0,0.625,24.0,0.75000,Real_Annual,C&L 01Aberdeen CityReal_Annual


## Create Previous and First Row Dictionaries

### Sort Dataframe

In [28]:
data = data.copy(deep=True)
# Replace Month Names with Values For Sorting purposes
data['Period'] = data.Period.replace({'January': '01', 'February': '02', 'March': '03', 'April': '04', 'May': '05','June': '06', 'July': '07', 'August': '08', 'September': '09', 'October': '10', 'November': '11', 'December': '12'}, regex=True)
data.sort_values(by=['DataType', 'LocalAuthority', 'Code', 'Period'], inplace=True)

displaywithHeader(data,'data')

,Key_CodePeriodDType,Key_CodePeriodFamilyGroupDType,Key_CodePeriodLADType,Code,LocalAuthority,Period,Value,Numerator,Denominator,FamilyRank,FamilyPct,ScotRank,ScotPct,DataType,Iter_CodeLADType
64334,C&L 012010-11Cash_Annual,C&L 012010-11Family Group 4Cash_Annual,C&L 012010-11Aberdeen CityCash_Annual,C&L 01,Aberdeen City,2010-11,0.33,637000.0,1922292.0,1.0,0.125,1.0,0.03125,Cash_Annual,C&L 01Aberdeen CityCash_Annual
64335,C&L 012011-12Cash_Annual,C&L 012011-12Family Group 4Cash_Annual,C&L 012011-12Aberdeen CityCash_Annual,C&L 01,Aberdeen City,2011-12,0.76,1554000.0,2045051.0,1.0,0.125,1.0,0.03125,Cash_Annual,C&L 01Aberdeen CityCash_Annual
64336,C&L 012012-13Cash_Annual,C&L 012012-13Family Group 4Cash_Annual,C&L 012012-13Aberdeen CityCash_Annual,C&L 01,Aberdeen City,2012-13,3.64,7883000.0,2163756.0,5.0,0.625,24.0,0.75000,Cash_Annual,C&L 01Aberdeen CityCash_Annual


### Define Variables for Loops

In [29]:
Previouss = []
Previous = None
Firsts = []
First = None
FirstSave = None
LocalAuthority = ""
DataType = ""
Code = ""
Period = ""
Value = ""
Numerator = ""
Denominator = ""
ScotRank = ""
ScotPct = ""
FamilyRank = ""
FamilyPct = ""
Iter_CodeLADType = ""

### Loop 'data' Dataframe and Record Previous and Firsts into List of Dictionaries

In [30]:
for row in data.itertuples():
    # If the curently stored Local_Authority and Code are both equal to the current row then this is not the first row for this indicator and local authority combination. As such Previous is calculated using all of the currently stored values in the variables (these are written to at the end of each loop) and First is populated using the stored dictionary in First_Save
    if Iter_CodeLADType == row.Iter_CodeLADType :
        Previous = {
            'Value': Value,
            'Numerator': Numerator,
            'Denominator': Denominator,
            'ScotRank': ScotRank,
            'ScotPct': ScotPct,
            'FamilyRank': FamilyRank,
            'FamilyPct': FamilyPct
        }
        First = FirstSave

    # If the curently stored Local_Authority and Code are both not equal to the current row then this is the first row for this indicator and local authority combination. as such the Previous object is set to None and the First object is populated using this rows values.
    elif Iter_CodeLADType != row.Iter_CodeLADType :
        FirstSave = {
            'Value': row.Value,
            'Numerator': row.Numerator,
            'Denominator': row.Denominator,
            'ScotRank': row.ScotRank,
            'ScotPct': row.ScotPct,
            'FamilyRank': row.FamilyRank,
            'FamilyPct': row.FamilyPct
        }
        First = None
        Previous = None

    # Append the First and Previous into their respective list variables.
    Previouss.append(Previous)
    Firsts.append(First)

    # Set all other variables to their respective columns values in the current row. This is used to both evaluate the if criteria above and to populate the next previous object.
    Data_Type = row.DataType
    Local_Authority = row.LocalAuthority
    Code = row.Code
    Period = row.Period
    Value = row.Value
    Numerator = row.Numerator
    Denominator = row.Denominator
    ScotRank = row.ScotRank
    ScotPct = row.ScotPct
    FamilyRank = row.FamilyRank
    FamilyPct = row.FamilyPct
    Iter_CodeLADType = row.Iter_CodeLADType

### Assign List Variables to Columns in 'data' Dataframe

In [31]:
data['Previous_Row'] = Previouss
data['First_Row'] = Firsts

displaywithHeader(data,'data')

,Key_CodePeriodDType,Key_CodePeriodFamilyGroupDType,Key_CodePeriodLADType,Code,LocalAuthority,Period,Value,Numerator,Denominator,FamilyRank,FamilyPct,ScotRank,ScotPct,DataType,Iter_CodeLADType,Previous_Row,First_Row
64334,C&L 012010-11Cash_Annual,C&L 012010-11Family Group 4Cash_Annual,C&L 012010-11Aberdeen CityCash_Annual,C&L 01,Aberdeen City,2010-11,0.33,637000.0,1922292.0,1.0,0.125,1.0,0.03125,Cash_Annual,C&L 01Aberdeen CityCash_Annual,None,None
64335,C&L 012011-12Cash_Annual,C&L 012011-12Family Group 4Cash_Annual,C&L 012011-12Aberdeen CityCash_Annual,C&L 01,Aberdeen City,2011-12,0.76,1554000.0,2045051.0,1.0,0.125,1.0,0.03125,Cash_Annual,C&L 01Aberdeen CityCash_Annual,"{'Value': 0.33, 'Numerator': 637000.0, 'Denomi...","{'Value': 0.33, 'Numerator': 637000.0, 'Denomi..."
64336,C&L 012012-13Cash_Annual,C&L 012012-13Family Group 4Cash_Annual,C&L 012012-13Aberdeen CityCash_Annual,C&L 01,Aberdeen City,2012-13,3.64,7883000.0,2163756.0,5.0,0.625,24.0,0.75000,Cash_Annual,C&L 01Aberdeen CityCash_Annual,"{'Value': 0.76, 'Numerator': 1554000.0, 'Denom...","{'Value': 0.33, 'Numerator': 637000.0, 'Denomi..."


## Create Comparison to First and Previous Row Dictionaries

### Save Info as Dictionary

In [32]:
Info_dict = info.set_index('Code_Sortable').to_dict('index')
Info_dict = NocaseDict(Info_dict)

### Define Value Change Function

In [33]:
# There are two niche cases here. One where previous and current values are both 0 resulting in 0% in all cases. Another where only the previous value is 0 resulting in None being returned as it is not possible to calculate % change from 0. Having looked at the dataset this has only occured 3 times and only affects Orkney and Eilean Siar for CHN20b. Further to this changes in percentage indicators are calculated using 100 as a denominator rather than previous. This is to avoid situations where very small percentages return 1000% or more change (which for our purposes seemed unreasonable to report).
def PercentChange_AimAdjusted(Previous, Current, Code):
    Aim = None
    SignedChange = None
    PercentChange = None
    GoldiMid = None
    IsPercentage = False

    if Previous == 0 and Current == 0:
        PercentChange = 0

    indicatorInfo = Info_dict[Code]
    Aim = indicatorInfo['Ranking_Type']
    GoldiMid = indicatorInfo['Ranking_GoldilocksMidpoint']
    IsPercentage = indicatorInfo['MeasureType'] == 'Percentage'

    if IsPercentage != True and Previous != 0:
        if Aim == "Descending":
            SignedChange = distance(Current, Previous)
            PercentChange = SignedChange/Previous
        if Aim == "Ascending":
            SignedChange = -distance(Current, Previous)
            PercentChange = SignedChange/Previous
        if Aim == "Goldilocks":
            Current_DistGoldi = abs(distance(Current, GoldiMid))
            Previous_DistGoldi = abs(distance(Previous, GoldiMid))
            SignedChange = distance(Previous_DistGoldi, Current_DistGoldi)
            PercentChange = SignedChange/Previous_DistGoldi
            
    elif IsPercentage == True and Previous != 0:
        if Aim == "Descending":
            PercentChange = distance(Current, Previous)
        if Aim == "Ascending":
            PercentChange = -distance(Current, Previous)
        if Aim == "Goldilocks":
            Current_DistGoldi = abs(distance(Current, GoldiMid))
            Previous_DistGoldi = abs(distance(Previous, GoldiMid))
            PercentChange = distance(Previous_DistGoldi, Current_DistGoldi)

    return PercentChange

### Define Row Comparison Function

In [34]:
def Changes(df):

    # Set the intial value of the return variable to none. This allows us to test to see if there were any changes present for a row and then return None instead of a dictionary of None values if not.
    Changes = None

    # Define all variables that will contain all of the relevant changes for a row.
    ScotRank_ChangeSincePrevious = None
    ScotPct_ChangeSincePrevious = None
    FamilyRank_ChangeSincePrevious = None
    FamilyPct_ChangeSincePrevious = None
    Value_ChangeSincePrevious = None
    Numerator_ChangeSincePrevious = None
    Denominator_ChangeSincePrevious = None
    ScotRank_ChangeSinceFirst = None
    ScotPct_ChangeSinceFirst = None
    FamilyRank_ChangeSinceFirst = None
    FamilyPct_ChangeSinceFirst = None
    Value_ChangeSinceFirst = None
    Numerator_ChangeSinceFirst = None
    Denominator_ChangeSinceFirst = None
    PercentChange_AimAdjusted_SincePrevious = None
    PercentChange_AimAdjusted_SinceFirst = None

    # If the value currently in Previous_Row is not None then there exists a previous object to calculate the changes using.
    if df['Previous_Row'] != None:
        # Calculate all differences by comparing the current rows value to the same columns value in the Previous_Row dictionary. Ranks and Percentiles are always positive so the calculations are more simple. The other values use the distance function defined at the start of the notebook to determine the signed difference between the values (comparing a current value of -1 to a previous value of 2 will result in -3 difference.)
        ScotRank_ChangeSincePrevious = - (df['ScotRank'] - df['Previous_Row'].get('ScotRank'))
        ScotPct_ChangeSincePrevious = - (df['ScotPct'] - df['Previous_Row'].get('ScotPct'))
        FamilyRank_ChangeSincePrevious = - (df['FamilyRank'] - df['Previous_Row'].get('FamilyRank'))
        FamilyPct_ChangeSincePrevious = - (df['FamilyPct'] - df['Previous_Row'].get('FamilyPct'))
        Value_ChangeSincePrevious = distance(df['Value'], df['Previous_Row'].get('Value'))
        Numerator_ChangeSincePrevious = distance(df['Numerator'], df['Previous_Row'].get('Numerator'))
        Denominator_ChangeSincePrevious = distance(df['Denominator'], df['Previous_Row'].get('Denominator'))
        PercentChange_AimAdjusted_SincePrevious = PercentChange_AimAdjusted(df['Previous_Row'].get('Value'), df['Value'], df['Code'])
        #Set Changes to true to avoid creating a dictionary of None values
        Changes = True

    # If the value currently in First_Row is not None then there exists a previous object to calculate the changes using.
    if df['First_Row'] != None:
        # Calculate all differences by comparing the current rows value to the same columns value in the First_Row dictionary. Ranks and Percentiles are always positive so the calculations are more simple. The other values use the distance function defined at the start of the notebook to determine the signed difference between the values (comparing a current value of -1 to a previous value of 2 will result in -3 difference.)
        ScotRank_ChangeSinceFirst = - (df['ScotRank'] - df['First_Row'].get('ScotRank'))
        ScotPct_ChangeSinceFirst = - (df['ScotPct'] - df['First_Row'].get('ScotPct'))
        FamilyRank_ChangeSinceFirst = - (df['FamilyRank'] - df['First_Row'].get('FamilyRank'))
        FamilyPct_ChangeSinceFirst = - (df['FamilyPct'] - df['First_Row'].get('FamilyPct'))
        Value_ChangeSinceFirst = distance(df['Value'], df['First_Row'].get('Value'))
        Numerator_ChangeSinceFirst = distance(df['Numerator'], df['First_Row'].get('Numerator'))
        Denominator_ChangeSinceFirst = distance(df['Denominator'], df['First_Row'].get('Denominator'))
        PercentChange_AimAdjusted_SinceFirst = PercentChange_AimAdjusted(df['First_Row'].get('Value'), df['Value'], df['Code'])
        #Set Changes to true to avoid creating a dictionary of None values
        Changes = True

    # If there were changes recorded in the previous steps then write these changes into a python dictionary and assign this to Changes
    if Changes != None:
        Changes = {
            "ScotRank_ChangeSincePrevious": ScotRank_ChangeSincePrevious,
            "ScotPct_ChangeSincePrevious": ScotPct_ChangeSincePrevious,
            "FamilyRank_ChangeSincePrevious": FamilyRank_ChangeSincePrevious,
            "FamilyPct_ChangeSincePrevious": FamilyPct_ChangeSincePrevious,
            "ScotRank_ChangeSinceFirst": ScotRank_ChangeSinceFirst,
            "ScotPct_ChangeSinceFirst": ScotPct_ChangeSinceFirst,
            "FamilyRank_ChangeSinceFirst": FamilyRank_ChangeSinceFirst,
            "FamilyPct_ChangeSinceFirst": FamilyPct_ChangeSinceFirst,
            "Value_ChangeSincePrevious": Value_ChangeSincePrevious,
            "Numerator_ChangeSincePrevious": Numerator_ChangeSincePrevious,
            "Denominator_ChangeSincePrevious": Denominator_ChangeSincePrevious,
            "Value_ChangeSinceFirst": Value_ChangeSinceFirst,
            "Numerator_ChangeSinceFirst": Numerator_ChangeSinceFirst,
            "Denominator_ChangeSinceFirst": Denominator_ChangeSinceFirst,
            "PercentChange_AimAdjusted_SincePrevious": PercentChange_AimAdjusted_SincePrevious,
            "PercentChange_AimAdjusted_SinceFirst": PercentChange_AimAdjusted_SinceFirst
        }

    return Changes

### Apply Functions Above

In [35]:
data['Changes'] = data.apply(Changes, axis=1)

displaywithHeader(data,'data')

,Key_CodePeriodDType,Key_CodePeriodFamilyGroupDType,Key_CodePeriodLADType,Code,LocalAuthority,Period,Value,Numerator,Denominator,FamilyRank,FamilyPct,ScotRank,ScotPct,DataType,Iter_CodeLADType,Previous_Row,First_Row,Changes
64334,C&L 012010-11Cash_Annual,C&L 012010-11Family Group 4Cash_Annual,C&L 012010-11Aberdeen CityCash_Annual,C&L 01,Aberdeen City,2010-11,0.33,637000.0,1922292.0,1.0,0.125,1.0,0.03125,Cash_Annual,C&L 01Aberdeen CityCash_Annual,None,None,None
64335,C&L 012011-12Cash_Annual,C&L 012011-12Family Group 4Cash_Annual,C&L 012011-12Aberdeen CityCash_Annual,C&L 01,Aberdeen City,2011-12,0.76,1554000.0,2045051.0,1.0,0.125,1.0,0.03125,Cash_Annual,C&L 01Aberdeen CityCash_Annual,"{'Value': 0.33, 'Numerator': 637000.0, 'Denomi...","{'Value': 0.33, 'Numerator': 637000.0, 'Denomi...","{'ScotRank_ChangeSincePrevious': -0.0, 'ScotPc..."
64336,C&L 012012-13Cash_Annual,C&L 012012-13Family Group 4Cash_Annual,C&L 012012-13Aberdeen CityCash_Annual,C&L 01,Aberdeen City,2012-13,3.64,7883000.0,2163756.0,5.0,0.625,24.0,0.75000,Cash_Annual,C&L 01Aberdeen CityCash_Annual,"{'Value': 0.76, 'Numerator': 1554000.0, 'Denom...","{'Value': 0.33, 'Numerator': 637000.0, 'Denomi...","{'ScotRank_ChangeSincePrevious': -23.0, 'ScotP..."


## Convert Dictionary Columns to JSON Columns

In [36]:
def PreviousConvertToJson(df):
    Previous_Row = simplejson.dumps(df['Previous_Row'], ignore_nan=True)
    return Previous_Row


def FirstConvertToJson(df):
    First_Row = simplejson.dumps(df['First_Row'], ignore_nan=True)
    return First_Row


def ChangesConvertToJson(df):
    Changes = simplejson.dumps(df['Changes'], ignore_nan=True)
    return Changes

data['Previous_Row'] = data.apply(PreviousConvertToJson, axis=1)
data['First_Row'] = data.apply(FirstConvertToJson, axis=1)
data['Changes'] = data.apply(ChangesConvertToJson, axis=1)

displaywithHeader(data,'data')

,Key_CodePeriodDType,Key_CodePeriodFamilyGroupDType,Key_CodePeriodLADType,Code,LocalAuthority,Period,Value,Numerator,Denominator,FamilyRank,FamilyPct,ScotRank,ScotPct,DataType,Iter_CodeLADType,Previous_Row,First_Row,Changes
64334,C&L 012010-11Cash_Annual,C&L 012010-11Family Group 4Cash_Annual,C&L 012010-11Aberdeen CityCash_Annual,C&L 01,Aberdeen City,2010-11,0.33,637000.0,1922292.0,1.0,0.125,1.0,0.03125,Cash_Annual,C&L 01Aberdeen CityCash_Annual,null,null,null
64335,C&L 012011-12Cash_Annual,C&L 012011-12Family Group 4Cash_Annual,C&L 012011-12Aberdeen CityCash_Annual,C&L 01,Aberdeen City,2011-12,0.76,1554000.0,2045051.0,1.0,0.125,1.0,0.03125,Cash_Annual,C&L 01Aberdeen CityCash_Annual,"{""Value"": 0.33, ""Numerator"": 637000.0, ""Denomi...","{""Value"": 0.33, ""Numerator"": 637000.0, ""Denomi...","{""ScotRank_ChangeSincePrevious"": -0.0, ""ScotPc..."
64336,C&L 012012-13Cash_Annual,C&L 012012-13Family Group 4Cash_Annual,C&L 012012-13Aberdeen CityCash_Annual,C&L 01,Aberdeen City,2012-13,3.64,7883000.0,2163756.0,5.0,0.625,24.0,0.75000,Cash_Annual,C&L 01Aberdeen CityCash_Annual,"{""Value"": 0.76, ""Numerator"": 1554000.0, ""Denom...","{""Value"": 0.33, ""Numerator"": 637000.0, ""Denomi...","{""ScotRank_ChangeSincePrevious"": -23.0, ""ScotP..."


## Separate Latest Values into Dataframe

In [37]:
LatestValues = data.copy(deep=True)
LatestValues.sort_values(by=['DataType','LocalAuthority', 'Code', 'Period'], inplace=True)
LatestValues = LatestValues.groupby(['DataType', 'LocalAuthority', 'Code']).tail(1)
LatestValues = LatestValues[
                            ['Key_CodePeriodDType',
                             'Key_CodePeriodLADType',
                             'Key_CodePeriodFamilyGroupDType',
                             'DataType',
                             'LocalAuthority',
                             'Code',
                             'Period',
                             'Value',
                             'Numerator',
                             'Denominator',
                             'ScotRank',
                             'ScotPct', 
                             'FamilyRank',
                             'FamilyPct',
                             'Previous_Row',
                             'First_Row', 
                             'Changes']
                            ]

displaywithHeader(LatestValues,'LatestValues')

,Key_CodePeriodDType,Key_CodePeriodLADType,Key_CodePeriodFamilyGroupDType,DataType,LocalAuthority,Code,Period,Value,Numerator,Denominator,ScotRank,ScotPct,FamilyRank,FamilyPct,Previous_Row,First_Row,Changes
64348,C&L 012024-25Cash_Annual,C&L 012024-25Aberdeen CityCash_Annual,C&L 012024-25Family Group 4Cash_Annual,Cash_Annual,Aberdeen City,C&L 01,2024-25,2.13,4335000.0,2031513.0,7.0,0.218750,1.0,0.125,"{""Value"": 3.01, ""Numerator"": 5956000.0, ""Denom...","{""Value"": 0.33, ""Numerator"": 637000.0, ""Denomi...","{""ScotRank_ChangeSincePrevious"": 4.0, ""ScotPct..."
64825,C&L 022024-25Cash_Annual,C&L 022024-25Aberdeen CityCash_Annual,C&L 022024-25Family Group 4Cash_Annual,Cash_Annual,Aberdeen City,C&L 02,2024-25,2.18,3739000.0,1716955.0,10.0,0.312500,3.0,0.375,"{""Value"": 2.31, ""Numerator"": 4061000.0, ""Denom...","{""Value"": 3.6, ""Numerator"": 5695000.0, ""Denomi...","{""ScotRank_ChangeSincePrevious"": 1.0, ""ScotPct..."
65212,C&L 032024-25Cash_Annual,C&L 032024-25Aberdeen CityCash_Annual,C&L 032024-25Family Group 4Cash_Annual,Cash_Annual,Aberdeen City,C&L 03,2024-25,3.01,4259000.0,1412803.0,14.0,0.518519,3.0,0.375,"{""Value"": 3.53, ""Numerator"": 4880000.0, ""Denom...","{""Value"": 5.02, ""Numerator"": 3528000.0, ""Denom...","{""ScotRank_ChangeSincePrevious"": -1.0, ""ScotPc..."


## Error Checks

In [38]:
# Default position is to assume checks are passed. If any of the checks (excluding numerator denominator checks) are failed below this will be changed and the final csv's will not be output. The numerator denominator errors should be checked at each refresh. The known errors (which exist within the LGBF raw data file) will be identified in the readme in the Error Outputs folder.
ChecksFailed = False
OutputText = []
# ScotRank should be between 32 and 1 and should not contain any NaN values
if not data['ScotRank'].between(1, 32).any() or data['ScotRank'].isnull().values.any():

    ChecksFailed = True
    maxrank = str(max(data['ScotRank']))
    minrank = str(min(data['ScotRank']))
    countnull = str(data['ScotRank'].isna().sum())

    OutputText.append(f'ScotRank checks failed : Max - {maxrank}, Min - {minrank}, Count of Null - {countnull}')

# FamilyRank should be between 8 and 1 and should not contain any NaN values
if not data['FamilyRank'].between(1, 8).any() or data['FamilyRank'].isnull().values.any():

    ChecksFailed = True
    maxrank = str(max(data['FamilyRank']))
    minrank = str(min(data['FamilyRank']))
    countnull = str(data['FamilyRank'].isna().sum())

    print(f'FamilyRank checks failed : Max - {maxrank}, Min - {minrank}, Count of Null - {countnull}')

# Code, Local_Authority, Period, Value should not contain any null values
if data[['Code', 'LocalAuthority', 'Period', 'Value']].isnull().values.any():

    ChecksFailed = True
    countnullCode = str(data['Code'].isna().sum())
    countnullLocal_Authority = str(data['LocalAuthority'].isna().sum())
    countnullPeriod = str(data['Period'].isna().sum())
    countnullReal_Value = str(data['Value'].isna().sum())

    OutputText.append(f'Null values found : Code - {countnullCode}, LocalAuthority - {countnullLocal_Authority}, Period - {countnullPeriod}, Value - {countnullReal_Value}')

# Value should equal numerator/denominator for both cash and real - These errors have been checked and exist in the original raw data file.
NumDenCheck = data.copy(deep=True)
NumDenCheck = NumDenCheck[pd.notnull(NumDenCheck['Numerator'])]

NumDenDivide_Checks = []
NumDenDivide_Check = None
FailReferences = []

for row in NumDenCheck.itertuples():
    if row.Value == 0 or math.isnan(row.Denominator) or math.isnan(row.Numerator):
        NumDenDivide_Check = None
    else:
        if math.isclose(row.Numerator/row.Denominator, row.Value, rel_tol=0.02):
            NumDenDivide_Check = True
        else:
            NumDenDivide_Check = False
            FailReferences.append(row.DataType + ":" + row.Code + ":" + row.Period + ":" + row.LocalAuthority + ":" + str(row.Value) + ":" + str(row.Numerator) + ":" + str(row.Denominator))

    NumDenDivide_Checks.append(NumDenDivide_Check)

if False in NumDenDivide_Checks :
    OutputText.append("Numerator/Denominator values check failed : See Error Outputs for csv of failures")
    FailReferences = sorted(list(set(FailReferences)))
    FailReferences = pd.DataFrame([sub.split(":") for sub in FailReferences])
    FailReferences = FailReferences.rename(columns={0: 'Type', 1: 'Code', 2: 'Period', 3: 'Local Authority', 4: 'Value', 5: 'Numerator', 6: 'Denominator'})
    FailReferences.to_csv("Error Outputs//Numerator Denominator Fail References.csv", index=False, encoding='utf-8-sig')

print(OutputText, sep='\n')

['Numerator/Denominator values check failed : See Error Outputs for csv of failures']


### Separate Files for Export to Avoid Github File Size Issues

In [39]:
data_realAnnual = data.query('DataType == "Real_Annual"')
data_cashAnnual = data.query('DataType == "Cash_Annual"')

data_realQuarterly = data.query('DataType == "Real_Quarterly"')
data_cashQuarterly = data.query('DataType == "Cash_Quarterly"')

data_realMonthly = data.query('DataType == "Real_Monthly"')
data_cashMonthly = data.query('DataType == "Cash_Monthly"')

## Output Files

In [40]:
scotValues.to_csv('Data Files\\Scottish Values.csv',index=False, encoding='utf-8-sig')
scotAverages.to_csv('Data Files\\Scottish Averages.csv', index=False, encoding='utf-8-sig')
FGAverages.to_csv('Data Files\\Family Averages.csv',index=False, encoding='utf-8-sig')
data_realAnnual.to_csv('Data Files\\Indicator Data - Real - Annual.csv',index=False, encoding='utf-8-sig')
data_cashAnnual.to_csv('Data Files\\Indicator Data - Cash - Annual.csv',index=False, encoding='utf-8-sig')
data_realQuarterly.to_csv('Data Files\\Indicator Data - Real - Quarterly.csv',index=False, encoding='utf-8-sig')
data_cashQuarterly.to_csv('Data Files\\Indicator Data - Cash - Quarterly.csv',index=False, encoding='utf-8-sig')
data_realMonthly.to_csv('Data Files\\Indicator Data - Real - Monthly.csv',index=False, encoding='utf-8-sig')
data_cashMonthly.to_csv('Data Files\\Indicator Data - Cash - Monthly.csv',index=False, encoding='utf-8-sig')
LatestValues.to_csv("Data Files//Latest Values.csv", index = False, encoding='utf-8-sig')